# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")


In [ ]:
print('test')

### Run the cell below every time to activate the installed environment. 

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 16384

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["VLLM_ATTENTION_BACKEND"] = "FLASH_ATTN"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
# # Original Prompts

# SYSTEM_PROMPT_MATH = (
#     "You are an expert mathematician. Solve the problem step-by-step. "
#     "Put your final answer inside \\boxed{}. "
#     "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
#     "e.g. \\boxed{3, 7}."
# )

# SYSTEM_PROMPT_MCQ = (
#     "You are an expert mathematician. "
#     "Read the problem and the answer choices below, then select the single best answer. "
#     "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
# )

# # Iteration 1

# # SYSTEM_PROMPT_MATH = (
# #     "You are an expert mathematical solver. "
# #     "Your goal is to solve the problem accurately and systematically. "
# #     "First, think step-by-step: define the variables, identify the necessary formulas, "
# #     "and perform the calculations clearly. Do not skip intermediate algebraic or arithmetic steps. "
# #     "Once you have an initial answer, double-check your calculations, signs, and units "
# #     "to ensure accuracy. "
# #     "Finally, output your answer. Put your final answer inside \\boxed{}. "
# #     "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{},"
# #     "e.g. \\boxed{3, 7}."
# #     "Do not continue writing after the boxed answer. "
# #     "Do not put full sentences or units inside the box unless explicitly requested. "
# # )

# # SYSTEM_PROMPT_MCQ = (
# #     "You are an expert multiple-choice math solver. "
# #     "Solve the problem systematically by breaking down the math step-by-step. "
# #     "Do not skip intermediate reasoning or calculations. "
# #     "You may use the provided choices as constraints or check your work by eliminating "
# #     "impossible options based on logic or estimation. "
# #     "After computing your final result, match it explicitly to one of the given choices. "
# #     "Your very last line must be exactly one of:  \\boxed{A}, \\boxed{B}, \\boxed{C}, \\boxed{D}, or \\boxed{E}. "
# #     "Do not continue writing after the boxed letter. "
# #     "Do not box anything except the final correct letter."
# # )


# def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
#     """Return (system_prompt, user_prompt) for a question."""
#     if options:
#         labels    = [chr(65 + i) for i in range(len(options))]
#         opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
#         return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
#     return SYSTEM_PROMPT_MATH, question


# # Verify with samples
# for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
#     sys_p, usr_p = build_prompt(item["question"], item.get("options"))
#     print(f"── {label} user prompt (first 200 chars) ──")
#     print(usr_p[:200], "...\n")

In [ ]:
# # IMPROVED FEW SHOT EXAMPLE
# SYSTEM_PROMPT_MATH = """You are an expert mathematical reasoning model.

# Solve the problem carefully and correctly. Use concise reasoning: show the key steps, but do not over-explain.

# Critical output rules:
# - You MUST end with exactly one final answer line.
# - The final answer line MUST be in this exact format:
# Final answer: \\boxed{...}
# - If multiple answers are required, put them in the requested order inside one box separated by commas.
# - Do not write anything after the boxed final answer.
# - Make sure the boxed answer directly answers the question asked."""

# SYSTEM_PROMPT_MCQ = """You are an expert multiple-choice mathematical reasoning model.

# Solve the problem carefully. Compare your result against all answer choices and choose the single best option.

# Critical output rules:
# - You MUST end with exactly one final answer line.
# - The final answer line MUST be in this exact format:
# Final answer: \\boxed{A}
# - The box must contain only one uppercase option letter.
# - Do not write anything after the boxed final answer.
# - Make sure the chosen letter corresponds to the correct answer choice."""

# def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
#     """Return (system_prompt, user_prompt) for a question."""
#     if options:
#         labels = [chr(65 + i) for i in range(len(options))]
#         opts_text = "\n".join(
#             f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options)
#         )

#         few_shot = """Follow the answer format shown in these examples.

# Example 1:
# Question: What is 2 + 2?
# Options:
# A. 3
# B. 4
# C. 5
# D. 6
# Solution: 2 + 2 = 4, which matches option B.
# Final answer: \\boxed{B}

# Example 2:
# Question: If 3x = 12, what is x?
# Options:
# A. 2
# B. 3
# C. 4
# D. 6
# Solution: Dividing by 3 gives x = 4, which matches option C.
# Final answer: \\boxed{C}
# """

#         user_prompt = f"""{few_shot}

# Now solve the following problem.

# Question:
# {question}

# Options:
# {opts_text}"""

#         return SYSTEM_PROMPT_MCQ, user_prompt

#     few_shot = """Follow the answer format shown in these examples.

# Example 1:
# Question: Compute 15 + 27.
# Solution: 15 + 27 = 42.
# Final answer: \\boxed{42}

# Example 2:
# Question: Find x and y if x = 4 and y = 9.
# Solution: The requested values are x = 4 and y = 9.
# Final answer: \\boxed{4, 9}
# """

#     user_prompt = f"""{few_shot}

# Now solve the following problem.

# Question:
# {question}"""

#     return SYSTEM_PROMPT_MATH, user_prompt

# print('loaded')

In [ ]:
# SYSTEM_PROMPT_MATH = """You are an elite mathematical reasoning AI. 
# Solve the problem systematically using the following rigid structure:

# 1. Analysis: Briefly identify the core question, given variables, and the necessary mathematical concepts.
# 2. Step-by-Step Draft: Perform the calculations clearly. Do not skip intermediate steps.
# 3. Verification: Explicitly double-check your initial logic, look for arithmetic errors, and verify units/signs.
# 4. Final Answer: Output ONLY your final answer on a new line enclosed in \boxed{}.

# Rules for the Final Answer:
# - If multiple answers are required, separate them by commas inside a single box (e.g., \boxed{3, 7}).
# - Do not write a single word or punctuation mark after the \boxed{} output."""

# SYSTEM_PROMPT_MCQ = """You are an elite mathematical reasoning AI. 
# Solve the multiple-choice problem systematically using the following rigid structure:

# 1. Analysis: Identify the core question and necessary concepts.
# 2. Step-by-Step Draft: Calculate the solution independently without looking at the options first.
# 3. Verification & Matching: Double-check your calculations, then compare your proven result against the provided A-E options.
# 4. Final Answer: Output ONLY the single matching uppercase letter on a new line enclosed in \boxed{}.

# Rules for the Final Answer:
# - The box must contain exactly one letter (e.g., \boxed{C}).
# - Do not write a single word or punctuation mark after the \boxed{} output."""

# def build_prompt(question: str, options: list = None) -> tuple[str, str]:
#     """Return (system_prompt, user_prompt) for a question."""
#     if options:
#         labels = [chr(65 + i) for i in range(len(options))]
#         opts_text = "\n".join(
#             f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options)
#         )
        
#         # Wrapping the user payload in XML-style tags helps small models delineate prompt from data
#         user_prompt = f"<question>\n{question}\n</question>\n\n<options>\n{opts_text}\n</options>"
#         return SYSTEM_PROMPT_MCQ, user_prompt

#     user_prompt = f"<question>\n{question}\n</question>"
#     return SYSTEM_PROMPT_MATH, user_prompt

# print('loaded')

In [4]:
SYSTEM_PROMPT_MCQ = (
    "You are an expert at solving multiple-choice math problems. "
    "Read the problem and all answer choices carefully, then determine which single option is correct.\n\n"
    "OUTPUT FORMAT RULES (strict):\n"
    "1. After your reasoning, end with a line of the exact form:\n"
    "   Final answer: \\boxed{X}\n"
    "   where X is a single capital letter, with NO spaces, NO punctuation, NO \\text{}.\n"
    "2. Correct:  Final answer: \\boxed{A}    Final answer: \\boxed{F}    Final answer: \\boxed{J}\n"
    "3. WRONG:    \\boxed{ A }   \\boxed{A.}   \\boxed{\\text{A}}   \\boxed{Option C}   \\boxed{C is correct}\n"
    "4. Do not write anything after the boxed final answer."
)

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step.\n\n"
    "The question contains one or more [ANS] placeholders marking where answers go.\n\n"
    "OUTPUT FORMAT RULES (strict):\n"
    "1. After your reasoning, end with a line of the form:\n"
    "   Final answer: \\boxed{...} \\boxed{...} ...\n"
    "   with one \\boxed{} per [ANS], in order, separated by single spaces.\n"
    "2. Each \\boxed{} contains exactly ONE answer. Do NOT put multiple answers separated by commas in one box.\n"
    "3. If an answer itself contains commas (a point, interval, list), keep it inside ONE box: \\boxed{(3, 5)} is one answer.\n"
    "4. Do not write anything after the final \\boxed{}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        last = labels[-1]
        few_shot = (
            "Examples of the required format:\n\n"
            "Example 1:\n"
            "Question: What is 2 + 2?\n"
            "Options:\nA. 3\nB. 4\nC. 5\nD. 6\n"
            "Reasoning: 2 + 2 = 4, so option B.\n"
            "Final answer: \\boxed{B}\n\n"
            "Example 2:\n"
            "Question: If x = 3, what is x^2?\n"
            "Options:\nA. 6\nB. 8\nC. 9\nD. 12\n"
            "Reasoning: 3^2 = 9, so option C.\n"
            "Final answer: \\boxed{C}\n"
        )
        user = (
            f"{few_shot}\n"
            f"Now solve this problem. Choose one option (A through {last}).\n\n"
            f"Question:\n{question}\n\n"
            f"Options:\n{opts_text}"
        )
        return SYSTEM_PROMPT_MCQ, user

    n_blanks = max(1, question.count("[ANS]"))
    few_shot = (
        "Examples of the required format:\n\n"
        "Example 1 (one [ANS]):\n"
        "Question: Compute 15 + 27. [ANS]\n"
        "Reasoning: 15 + 27 = 42.\n"
        "Final answer: \\boxed{42}\n\n"
        "Example 2 (two [ANS] markers):\n"
        "Question: If x = 4 [ANS] and y = 9 [ANS], state both.\n"
        "Reasoning: x = 4 and y = 9.\n"
        "Final answer: \\boxed{4} \\boxed{9}\n\n"
        "Example 3 (answer contains a comma — stays in ONE box):\n"
        "Question: The point on the line is [ANS].\n"
        "Reasoning: The point is (3, 5).\n"
        "Final answer: \\boxed{(3, 5)}\n"
    )
    hint = (
        f"This problem has {n_blanks} [ANS] placeholder{'s' if n_blanks > 1 else ''}, "
        f"so output exactly {n_blanks} \\boxed{{}} block{'s' if n_blanks > 1 else ''} at the end."
    )
    user = f"{few_shot}\n{hint}\n\nQuestion:\n{question}"
    return SYSTEM_PROMPT_MATH, user

In [ ]:
# SYSTEM_PROMPT_MATH = (
#     "You are an expert mathematician. Solve the problem step-by-step. "
#     "The question contains one or more [ANS] placeholders marking where answers go. "
#     "Produce EXACTLY one \\boxed{} per [ANS], in the same order they appear, "
#     "with NOTHING between consecutive boxes except whitespace. "
#     "Example: \\boxed{42} \\boxed{x^2+1} \\boxed{(0, 1)}. "
#     "Do not put multiple answers inside a single \\boxed{}."
# )

# SYSTEM_PROMPT_MCQ = """You are an expert multiple-choice math problem solver.

# Read the problem and all answer choices carefully. Solve the problem and choose exactly one option.

# You MUST end your response with exactly one final answer line:
# Final answer: \\boxed{A}

# The box must contain only one uppercase option letter. Do not write anything after the boxed final answer."""

# def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
#     """Return (system_prompt, user_prompt) for a question."""
#     if options:
#         labels    = [chr(65 + i) for i in range(len(options))]
#         opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
#         return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
#     return SYSTEM_PROMPT_MATH, question


# # Verify with samples
# for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
#     sys_p, usr_p = build_prompt(item["question"], item.get("options"))
#     print(f"── {label} user prompt (first 200 chars) ──")
#     print(usr_p[:200], "...\n")


## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=True,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token

# tokenizer.padding_side = "left"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
# Build prompts for first 5 entries
prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=True)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

In [11]:
print('test')

### Generate with Transformers (for Datahub)

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:10]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!